In [ ]:
## This script is only for archetypes in samples where UMAP with HDBSCAN failed to capture specialist cell population robustly. This 
## graph based clustering algorithm will robustly identifies the specialist cell population.

In [1]:
library(glue)
sample <- "SMC25-T"  # Mention the sample name here

exp_data <- read.csv(glue("../../pipeline_testing/data/{sample}/{sample}_TPM_mod.csv"),row.names=1,check.names = FALSE)

##Choose the respective archetypes where you need reclustering and comment out others

# a2_cell <- read.table(glue("../../pipeline_testing/specialist_generalist/{sample}/Archetype_2_specialists.tsv"), sep="\t")
a3_cell <- read.table(glue("../../pipeline_testing/specialist_generalist/{sample}/Archetype_3_specialists.tsv"), sep="\t")
# a4_cell <- read.table(glue("../../pipeline_testing/specialist_generalist/{sample}/Archetype_4_specialists.tsv"), sep="\t")
# a5_cell <- read.table(glue("../../pipeline_testing/specialist_generalist/{sample}/Archetype_5_specialists.tsv"), sep="\t")

In [2]:
# total_cell <- rbind(a3_cell, a5_cell)
exp_data_filt <- exp_data[,colnames(exp_data) %in% a3_cell$x]
dim(exp_data_filt)

[1] 33694   195

## Seurat louvain graph based clustering

In [3]:
library(Seurat)
library(Matrix)

rownames(exp_data_filt) <- gsub("_", "-", rownames(exp_data_filt))
sparse_data <- as(as.matrix(exp_data_filt), "dgCMatrix")
sobj <- CreateSeuratObject(counts = sparse_data)
sobj <- SetAssayData(sobj, layer = "data", new.data = sparse_data)

# 5. Feature Selection
sobj <- FindVariableFeatures(sobj, selection.method = "vst", nfeatures = 2000)

# 6. Scaling
sobj <- ScaleData(sobj)

# 7. PCA
sobj <- RunPCA(sobj, npcs = 30)

# 8. Graph-Based Clustering 
sobj <- FindNeighbors(sobj, dims = 1:20)

sobj <- FindClusters(sobj, resolution = 0.3)

sobj <- RunUMAP(sobj, dims = 1:20)

dir.create("../../pipeline_testing/seurat_clustering/", recursive = TRUE)
dir.create(glue("../../pipeline_testing/seurat_clustering/{sample}_A3/"), recursive = TRUE)

pdf(file = glue("../../pipeline_testing/seurat_clustering/{sample}_A3/seurat_clustering_2d.pdf"))
DimPlot(sobj, reduction = "umap", label = TRUE)
dev.off()

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Finding variable features for layer counts

Centering and scaling data matrix

PC_ 1 
Positive:  SEZ6L2, CTD-3131K8.2, SPIRE2, ACAD9, SOD3, PLCG2, ATP10B, DHRS11, PRR15L, HIST1H4E 
	   POLD1, GCNT3, SLC39A5, ERN2, RP11-395B7.2, CTD-2186M15.3, FAM135A, XPOT, DUOXA2, SELENBP1 
	   LINC00467, MINPP1, BCAM, STEAP2, CTTN, TMEM41A, MUC4, CEACAM5, SMIM22, NAT9 
Negative:  TMSB4X, B2M, SRGN, EIF1, PTMA, EEF1A1, RPL9, BTG1, SARAF, CREM 
	   CD52, EEF1D, DDX5, CD3D, RPL36AL, FOS, CXCR4, DUSP1, CD44, ACTB 
	   YPEL5, RPL4, CIRBP, TSC22D3, H3F3B, TOMM7, CYBA, HLA-E, PTPRC, CALM1 
PC_ 2 
Positive:  MALAT1, BTG1, SRGN, DCP1B, C11orf71, FUT2, WIZ, ZCCHC3, PICK1, LANCL1 
	   GSAP, CCDC64B, VTA1, MTR, CAD, OGFOD2, DHRS4L2, WDR5B, MPZL1, SLC33A1 
	   PRMT3, IP6K1, ELAC2, AP4B1, KMT2B, SENP8, SPAG1, ZNF573, TLK2, ZCWPW1 
Negat

Modularity Optimizer version 1.3.0 by Ludo Waltman and Nees Jan van Eck

Number of nodes: 195
Number of edges: 7336

Running Louvain algorithm...
Maximum modularity in 10 random starts: 0.8154
Number of communities: 3
Elapsed time: 0 seconds


Warning message:
“The default method for RunUMAP has changed from calling Python UMAP via reticulate to the R-native UWOT using the cosine metric
To use Python UMAP via reticulate, set umap.method to 'umap-learn' and metric to 'correlation'
This message will be shown once per session”
23:13:44 UMAP embedding parameters a = 0.9922 b = 1.112

23:13:45 Read 195 rows and found 20 numeric columns

23:13:45 Using Annoy for neighbor search, n_neighbors = 30

23:13:45 Building Annoy index with metric = cosine, n_trees = 50

0%   10   20   30   40   50   60   70   80   90   100%

[----|----|----|----|----|----|----|----|----|----|

*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
|

23:13:45 Writing NN index file to temp file /tmp/RtmpiiFTjk/file33a2a33beb4eb2

23:13:45 Searching Annoy index using 1 thread, search_k = 3000

23:13:45 Annoy recall = 100%

23:13:45 Commencing smooth kNN distance calibration using 1 thread
 with target n_neighbors 

agg_record_33a2a36bec0a40 
                        2

In [4]:
cell_clusters <- data.frame(
  cells = rownames(sobj@meta.data),
  cluster = sobj$seurat_clusters
)
rownames(cell_clusters) <- NULL

In [5]:
# Split the dataframe by cluster
split_clusters <- split(cell_clusters, cell_clusters$cluster)

# Apply a function to save each split dataframe
lapply(names(split_clusters), function(cluster_id) {
  file_name <- paste0("Archetype_3_", cluster_id, "_specialists_cluster_", cluster_id, ".tsv")
  write.table(split_clusters[[cluster_id]]$cells, 
              file = glue("../../pipeline_testing/seurat_clustering/{sample}_A3/{file_name}"), 
              sep = "\t", 
              row.names = TRUE, 
              quote = FALSE)
})

[[1]]
NULL

[[2]]
NULL

[[3]]
NULL